In [43]:
import pandas as pd
import numpy as np

from mujoco_playground import registry
import mujoco

In [44]:
data_path = "../data/huggingface/G1_WB_Dex5_Pickup_Pillow.parquet"

## Explore dataset

In [45]:
df = pd.read_parquet(data_path)

In [46]:
df.columns

Index(['observation.state.ee_state', 'observation.state.hand_state',
       'observation.state.robot_q_current', 'action.ee_action',
       'action.hand_cmd', 'action.robot_q_desired', 'timestamp', 'frame_index',
       'episode_index', 'index', 'task_index'],
      dtype='str')

In [68]:
# From my understanding each episode is one repetition of tele-operation and data collection
# So the dataset contains many episodes which are repetitions of the same action
# We can filter for just episode 0
episode_0 = df[df["episode_index"] == 0]

In [69]:
# Notice that all actions have 36 dims
episode_0["action.robot_q_desired_len"] = episode_0["action.robot_q_desired"].apply(len)
episode_0["action.robot_q_desired_len"].unique()

array([36])

In [70]:
# Notice that all observations also have 36 dims
episode_0["observation.state.robot_q_current_len"] = episode_0["observation.state.robot_q_current"].apply(len)
episode_0["observation.state.robot_q_current_len"].unique()

array([36])

In [71]:
# Define a trajectory as the list of desired actions
trajectory = episode_0["action.robot_q_desired"]
trajectory_array = np.stack(trajectory.to_numpy())

## Explore the Unitree G1 model

In [73]:
# List of robots available
registry.locomotion.ALL_ENVS

('ApolloJoystickFlatTerrain',
 'BarkourJoystick',
 'BerkeleyHumanoidJoystickFlatTerrain',
 'BerkeleyHumanoidJoystickRoughTerrain',
 'G1JoystickFlatTerrain',
 'G1JoystickRoughTerrain',
 'Go1JoystickFlatTerrain',
 'Go1JoystickRoughTerrain',
 'Go1Getup',
 'Go1Handstand',
 'Go1Footstand',
 'H1InplaceGaitTracking',
 'H1JoystickGaitTracking',
 'Op3Joystick',
 'SpotFlatTerrainJoystick',
 'SpotGetup',
 'SpotJoystickGaitTracking',
 'T1JoystickFlatTerrain',
 'T1JoystickRoughTerrain')

In [76]:
env_name = "G1JoystickFlatTerrain"  # This is the Unitree G1 model
env = registry.load(env_name)
model = env.mj_model

# nq -> size of qpos ie the position vector of size 36.
# 1 for each of 29 joints 7 for base joint (3 for position + 4 for quaternion)
print(f"nq: {model.nq}")
# qv -> size of qvel the velocity vector of size 35
# This is 1 less that nq because the quaternion has only 3 degrees of freedom
print(f"nv: {model.nv}")
# nu -> number of actuators which is 29
# This is what we can control which is each of the 29 joints (except the floating base)
print(f"nu: {model.nu}")

for i in range(model.njnt):
    print(i, model.joint(i).name)

nq: 36
nv: 35
nu: 29
0 floating_base_joint
1 left_hip_pitch_joint
2 left_hip_roll_joint
3 left_hip_yaw_joint
4 left_knee_joint
5 left_ankle_pitch_joint
6 left_ankle_roll_joint
7 right_hip_pitch_joint
8 right_hip_roll_joint
9 right_hip_yaw_joint
10 right_knee_joint
11 right_ankle_pitch_joint
12 right_ankle_roll_joint
13 waist_yaw_joint
14 waist_roll_joint
15 waist_pitch_joint
16 left_shoulder_pitch_joint
17 left_shoulder_roll_joint
18 left_shoulder_yaw_joint
19 left_elbow_joint
20 left_wrist_roll_joint
21 left_wrist_pitch_joint
22 left_wrist_yaw_joint
23 right_shoulder_pitch_joint
24 right_shoulder_roll_joint
25 right_shoulder_yaw_joint
26 right_elbow_joint
27 right_wrist_roll_joint
28 right_wrist_pitch_joint
29 right_wrist_yaw_joint


In [77]:
# Looking at the trajectory confirms that we have 7 numbers for the floating base
# trajectory[0][0:3] -> [0., 0., 0.7853193]
# This is the x, y, z position at the start. z is non 0 as robot is stood up
# trajectory[0][3:7] -> [1., 0., 0., 0.]
# This is the quaternion for upright orientation
trajectory[0]

array([ 0.        ,  0.        ,  0.7853193 ,  1.        ,  0.        ,
        0.        ,  0.        ,  0.09639694, -0.05822653,  0.03272714,
       -0.087267  , -0.01099355,  0.04259708,  0.07875177, -0.02435293,
       -0.0984899 , -0.087267  ,  0.07469133,  0.08545893,  0.02255546,
       -0.04828033,  0.06914613, -0.0866328 ,  0.2989125 , -0.34025115,
        1.3659769 ,  0.1509939 , -0.0969459 , -0.03604558, -0.12574899,
       -0.30353016,  0.27248105,  1.3514031 ,  0.01900237, -0.12952945,
        0.01192439], dtype=float32)

## Apply trajectory to model

To see how to do this check scripts/parquet_player.py